# 03 — Exposure Labels

Identifies anchor posts and classifies users as exposed/unexposed.

**Anchor post definition:** keyword filter + BART zero-shot NLI confirms a negative admissions
outcome (`facebook/bart-large-mnli`). SVM scores are computed and saved for characterisation —
they are **not** used to select anchors, keeping the treatment definition independent of the
outcome model (prevents circularity with NB04/NB06).

**Exposed user:** commented on an anchor thread during the Sep–Nov anchor window.
`exposure_intensity` = max BART top-negative-label score across anchor threads commented on.
`exposure_prob` = log-normalised upvote score of the most popular anchor thread commented on.

**Unexposed user:** active the same week as anchor events, never commented on an anchor thread.

**Outputs:** `anchor_posts.parquet`, `exposure_labels.parquet`

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                     STUDY CONFIG                            ║
# ║  Change only this cell to run on a different dataset        ║
# ╚══════════════════════════════════════════════════════════════╝

STUDY_ID   = 'gradadmissions'
SUBREDDIT  = 'gradadmissions'   # change to 'mscs' for MSCS pipeline

# Anchor period cycles — numbered chronologically.
#   Cycle 1: Aug 2022 – May 2023 admission cycle
#   Cycle 2: Aug 2023 – May 2024 admission cycle
#   Cycle 3: Aug 2024 – May 2025 admission cycle
CYCLES = {
    1: {'anchor_start': '2022-09-01', 'anchor_end': '2022-11-30',
        'active_start': '2022-08-01', 'active_end':  '2023-05-31'},
    2: {'anchor_start': '2023-09-01', 'anchor_end': '2023-11-30',
        'active_start': '2023-08-01', 'active_end':  '2024-05-31'},
    3: {'anchor_start': '2024-09-01', 'anchor_end': '2024-11-30',
        'active_start': '2024-08-01', 'active_end':  '2025-05-31'},
}

# Keyword filter — edit for a different community
NEGATIVE_KEYWORDS = [
    r'\breject(?:ed|ion)\b',     r'\bdeclin(?:ed|ing)\b',
    r'\bwaitlist(?:ed)?\b',       r'\bfunding\s+(?:lost|cut|removed|denied|gap|issue)\b',
    r'\bno\s+funding\b',          r'\bstipend\b',
    r'\bwithdrew?\s+(?:offer|admission)\b', r'\bacceptance\s+rate\b',
    r'\bno\s+(?:offer|response|interview)\b', r'\bsilence\s+from\b',
    r'\bnot\s+(?:accepted|admitted|selected)\b', r'\bgave\s+up\b',
    r'\bmental\s+health\b',        r'\banxi(?:ous|ety)\b',
    r'\bdepress(?:ed|ing|ion)\b',  r'\bstress(?:ed|ful)?\b',
    r'\boverwhelm(?:ed|ing)\b',    r'\bscared\b',
    r'\bworr(?:ied|ying)\b',       r'\bfalling\s+apart\b',
    r'\bbreaking\s+down\b',        r"\bcan(?:'t|not)\s+(?:take|handle|cope)\b",
    r'\bno\s+chance\b',            r'\bnot\s+good\s+enough\b',
    r'\bregret\b', r'\bfailed\b',  r'\bimposter\b',
]

# BART NLI anchor classification — anchor = keyword match AND bart_is_negative == True
CANDIDATE_LABELS = [
    'negative admissions outcome',
    'rejection or funding loss',
    'giving up on graduate school',
    'general admissions discussion',   # ← negative class (not an anchor label)
]
BART_BATCH_SIZE = 16
MAX_CHARS       = 1024   # truncate texts fed to BART

# Popularity weighting: 1.0=full log-normalised upvote score, 0.0=uniform
POPULARITY_WEIGHT = 1.0

In [ ]:
import json, re
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'data' / 'processed' / SUBREDDIT
MODEL_DIR = ROOT / 'models'
POSTS_PATH    = DATA_DIR / 'posts_clean.jsonl'
COMMENTS_PATH = DATA_DIR / 'comments_clean.jsonl'

keyword_pattern = re.compile('|'.join(NEGATIVE_KEYWORDS), re.IGNORECASE)

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line: rows.append(json.loads(line))
    return rows

print(f'Subreddit: r/{SUBREDDIT} | Study: {STUDY_ID}')
print(f'Cycles: {list(CYCLES.keys())}')
print(f'Anchor method: keyword filter + BART NLI (facebook/bart-large-mnli)')

## 1) Load raw posts


In [16]:
raw_posts = load_jsonl(POSTS_PATH)
posts = pd.DataFrame([{
    'id':           r['id'],
    'author':       r['author'],
    'created_dt':   pd.Timestamp(r['created_dt']),
    'clean_text':   r.get('clean_text', ''),
    'score':        r.get('score', 0),
    'num_comments': r.get('num_comments', 0),
} for r in raw_posts])
print(f'Posts loaded: {len(posts):,} from {posts["author"].nunique():,} authors')
print(f'Date range: {posts["created_dt"].min().date()} → {posts["created_dt"].max().date()}')


Posts loaded: 17,200 from 6,709 authors
Date range: 2023-08-01 → 2025-07-30


## 2) Filter to anchor periods and score with SVM classifiers


In [17]:
def assign_cycle(dt):
    for cycle, w in CYCLES.items():
        if pd.Timestamp(w['anchor_start'], tz='UTC') <= dt <= pd.Timestamp(w['anchor_end'] + ' 23:59:59', tz='UTC'):
            return cycle
    return None

posts['cycle'] = posts['created_dt'].apply(assign_cycle)
anchor_candidates = posts[posts['cycle'].notna()].copy()
print(f'Posts in anchor periods: {len(anchor_candidates):,}')
print(anchor_candidates['cycle'].value_counts().sort_index())


Posts in anchor periods: 3,525
cycle
1.0    1279
2.0    2246
Name: count, dtype: int64


In [18]:
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
print("Loaded anxiety")
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
print("Loaded Depression")
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Loaded stress.')

def sigmoid(x): return 1 / (1 + np.exp(-x))

texts = anchor_candidates['clean_text'].tolist()
anchor_candidates['anx_score'] = sigmoid(clf_anx.decision_function(texts))
anchor_candidates['dep_score'] = sigmoid(clf_dep.decision_function(texts))
anchor_candidates['str_score'] = sigmoid(clf_str.decision_function(texts))
anchor_candidates['mean_mh_score'] = anchor_candidates[['anx_score','dep_score','str_score']].mean(axis=1)
print(f'Scored {len(anchor_candidates):,} anchor-period posts')


Loaded anxiety
Loaded Depression
Loaded stress.
Scored 3,525 anchor-period posts


## 3) Keyword filter + BART NLI → anchor posts

In [ ]:
# ── Keyword filter ──────────────────────────────────────────────────────────
anchor_candidates['has_neg_keyword'] = anchor_candidates['clean_text'].str.contains(
    keyword_pattern, na=False
)
kw_filtered = anchor_candidates[anchor_candidates['has_neg_keyword']].copy()
print(f'Posts in anchor periods:  {len(anchor_candidates):,}')
print(f'After keyword filter:     {len(kw_filtered):,}')
print(kw_filtered['cycle'].value_counts().sort_index())

# ── Load BART NLI pipeline ──────────────────────────────────────────────────
import torch
from transformers import pipeline as hf_pipeline

device = 0 if torch.cuda.is_available() else -1
print(f'\nDevice: {"GPU" if device == 0 else "CPU"}')
nli_pipe = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=device,
)
print('BART NLI pipeline loaded.')

# ── Run BART inference on keyword-filtered candidates ───────────────────────
def parse_result(r):
    top_label   = r['labels'][0]
    top_score   = float(r['scores'][0])
    label_score = dict(zip(r['labels'], r['scores']))
    neg_labels  = [l for l in r['labels'] if l != 'general admissions discussion']
    top_neg_label = max(neg_labels, key=lambda l: label_score[l])
    top_neg_score = float(label_score[top_neg_label])
    return {
        'bart_top_label':     top_label,
        'bart_top_score':     top_score,
        'bart_top_neg_label': top_neg_label,
        'bart_top_neg_score': top_neg_score,
        'bart_is_negative':   top_label != 'general admissions discussion',
    }

texts = kw_filtered['clean_text'].fillna('').str[:MAX_CHARS].tolist()
print(f'\nRunning BART NLI on {len(texts):,} keyword-filtered posts...')

raw_results = []
for i in range(0, len(texts), BART_BATCH_SIZE):
    batch = texts[i : i + BART_BATCH_SIZE]
    out   = nli_pipe(batch, candidate_labels=CANDIDATE_LABELS, multi_label=False)
    if isinstance(out, dict):
        out = [out]
    raw_results.extend(out)
    if i > 0 and i % (BART_BATCH_SIZE * 20) == 0:
        print(f'  [{i:,}/{len(texts):,}]')

kw_filtered = kw_filtered.join(
    pd.DataFrame([parse_result(r) for r in raw_results], index=kw_filtered.index)
)
print(f'Inference complete. Label distribution:')
print(kw_filtered['bart_top_label'].value_counts())

In [ ]:
# ── Select anchors: keyword match + BART confirms negative outcome ──────────
anchor_posts = kw_filtered[kw_filtered['bart_is_negative']].copy()
print(f'Anchor posts (keyword + BART negative): {len(anchor_posts):,}')
print(anchor_posts['cycle'].value_counts().sort_index())
print('\nBART label distribution among anchors:')
print(anchor_posts['bart_top_label'].value_counts())

In [ ]:
anchor_posts[['id','author','created_dt','cycle','clean_text',
              'anx_score','dep_score','str_score','mean_mh_score',
              'bart_top_label','bart_top_score',
              'bart_top_neg_label','bart_top_neg_score','bart_is_negative',
              'score','num_comments']]\
    .to_parquet(DATA_DIR / 'anchor_posts.parquet', index=False)
print('Saved anchor_posts.parquet')

anchor_ids_by_cycle     = {c: set(anchor_posts[anchor_posts['cycle']==c]['id'])     for c in CYCLES}
anchor_authors_by_cycle = {c: set(anchor_posts[anchor_posts['cycle']==c]['author']) for c in CYCLES}
print(f'Anchor IDs — ' + ' | '.join(f'C{c}: {len(v):,}' for c, v in anchor_ids_by_cycle.items()))

## 4) Load comments → identify exposed users


In [21]:
import datetime
print('Loading comments...')
comment_rows = []
with open(COMMENTS_PATH) as f:
    for line in f:
        r = json.loads(line)
        comment_rows.append({
            'id':         r.get('id',''),
            'author':     r.get('author',''),
            'post_id':    r.get('post_id', ''),
            'created_dt': pd.Timestamp(r['created_dt']),
        })
comments = pd.DataFrame(comment_rows)
print(f'Comments loaded: {len(comments):,} from {comments["author"].nunique():,} authors')


Loading comments...
Comments loaded: 124,547 from 13,212 authors


In [22]:
all_anchor_ids = set().union(*anchor_ids_by_cycle.values())
anchor_comments = comments[comments['post_id'].isin(all_anchor_ids)].copy()
print(f'Comments on anchor posts: {len(anchor_comments):,}')

def comment_cycle(post_id):
    for c, ids in anchor_ids_by_cycle.items():
        if post_id in ids: return c
    return None

anchor_comments['cycle'] = anchor_comments['post_id'].apply(comment_cycle)
print(anchor_comments['cycle'].value_counts().sort_index())


Comments on anchor posts: 993
cycle
1    321
2    672
Name: count, dtype: int64


In [ ]:
global_max_log_score = np.log1p(anchor_posts['score'].clip(lower=0).max())
exposed_records = []

for cycle in CYCLES:
    cycle_comments = anchor_comments[anchor_comments['cycle'] == cycle]
    excluded = anchor_authors_by_cycle[cycle]
    eligible = cycle_comments[~cycle_comments['author'].isin(excluded)].copy()

    ap_scores = anchor_posts[['id','score','bart_top_neg_score']].rename(
        columns={'score': 'post_upvote_score', 'bart_top_neg_score': 'post_bart_score'})
    eligible = eligible.merge(ap_scores, left_on='post_id', right_on='id', how='left')
    eligible['post_bart_score']   = eligible['post_bart_score'].fillna(0.0)
    eligible['post_upvote_score'] = eligible['post_upvote_score'].fillna(0).clip(lower=0)

    author_max_bart  = eligible.groupby('author')['post_bart_score'].max()
    author_max_score = eligible.groupby('author')['post_upvote_score'].max()

    for author in author_max_bart.index:
        if POPULARITY_WEIGHT > 0 and global_max_log_score > 0:
            p_pop = np.log1p(author_max_score[author]) / global_max_log_score
        else:
            p_pop = 1.0
        exposed_records.append({
            'author':             author,
            'exposed':            True,
            'cycle':              cycle,
            'exposure_intensity': float(author_max_bart[author]),
            'exposure_prob':      float(p_pop),
        })
    print(f'Cycle {cycle} — exposed: {len(author_max_bart):,} '
          f'(excluded {len(set(cycle_comments["author"]) & excluded):,} anchor authors)')

exposed_df = pd.DataFrame(exposed_records)
print(f'\nTotal exposed: {len(exposed_df):,}')

## 5) Identify unexposed users


In [ ]:
def iso_week(dt): return dt.strftime('%G-W%V')
posts['iso_week']    = posts['created_dt'].apply(iso_week)
comments['iso_week'] = comments['created_dt'].apply(iso_week)
anchor_posts['iso_week'] = anchor_posts['created_dt'].apply(iso_week)

anchor_weeks_by_cycle = {c: set(anchor_posts[anchor_posts['cycle']==c]['iso_week']) for c in CYCLES}

unexposed_records = []
for cycle in CYCLES:
    anchor_weeks = anchor_weeks_by_cycle[cycle]
    exposed_this_cycle = set(exposed_df[exposed_df['cycle']==cycle]['author'])
    active = set(posts[posts['iso_week'].isin(anchor_weeks)]['author']) | \
             set(comments[comments['iso_week'].isin(anchor_weeks)]['author'])
    unexposed = active - exposed_this_cycle
    for author in unexposed:
        unexposed_records.append({'author': author, 'exposed': False, 'cycle': cycle,
                                  'exposure_intensity': 0.0, 'exposure_prob': 0.0})
    print(f'Cycle {cycle} — active: {len(active):,} | exposed: {len(exposed_this_cycle):,} | unexposed: {len(unexposed):,}')

unexposed_df = pd.DataFrame(unexposed_records)
print(f'\nTotal unexposed: {len(unexposed_df):,}')

## 6) Combine and save


In [ ]:
exposure_df = pd.concat([exposed_df, unexposed_df], ignore_index=True)
exposure_df['exposure_intensity'] = exposure_df['exposure_intensity'].fillna(0.0).astype(float)
exposure_df['exposure_prob']      = exposure_df['exposure_prob'].fillna(0.0).astype(float)

print(f'Total records: {len(exposure_df):,} | Unique users: {exposure_df["author"].nunique():,}')
print('\nExposed vs unexposed by cycle:')
print(exposure_df.groupby(['cycle','exposed']).size().unstack(fill_value=0))
print('\nExposure intensity summary (exposed only):')
print(exposure_df[exposure_df['exposed']].groupby('cycle')['exposure_intensity'].describe().round(3))
print('\nExposure prob summary (exposed only):')
print(exposure_df[exposure_df['exposed']].groupby('cycle')['exposure_prob'].describe().round(3))

exposure_df.to_parquet(DATA_DIR / 'exposure_labels.parquet', index=False)
print('\nSaved exposure_labels.parquet')
print('Columns:', exposure_df.columns.tolist())

## 7) Sanity checks


In [ ]:
n_cycles = len(CYCLES)
both_cycles = exposure_df.groupby('author')['cycle'].nunique()
print(f'Users in all {n_cycles} cycles: {(both_cycles==n_cycles).sum():,}')
exposed_both = exposure_df[exposure_df['exposed']].groupby('author')['cycle'].nunique()
print(f'Exposed in all {n_cycles} cycles: {(exposed_both==n_cycles).sum():,}')
print(f'\nexposure_prob range (exposed): {exposure_df[exposure_df["exposed"]]["exposure_prob"].agg(["min","max"]).round(3).to_dict()}')
print(f'Average exposure_prob among exposed: {exposure_df[exposure_df["exposed"]]["exposure_prob"].mean():.4f}')
